In [1]:
# Python libs
import pandas as pd

# Magics
from helpers import (
    load_sql_magic,
)
load_sql_magic()          # %%sql   — query DataFrames via duckdb (no extra installs)

True

In [2]:
import duckdb

DB_PATH = "./data/ab_events.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)
event_log = con.execute("SELECT * FROM events").df()
con.close()

print(event_log.shape)
event_log.head()

(1995151, 7)


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 10:32:46,2,6927017134761466251,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 10:33:32,2,6927017134761466251,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 21:27:17,4,1241045378047175561,US,"{""num01"":""b""}",page_view,NaN
3,2025-01-01 21:30:00,4,1241045378047175561,US,"{""num01"":""b""}",watch,NaN
4,2025-01-01 21:30:55,4,1241045378047175561,US,"{""num01"":""b""}",page_view,NaN


In [3]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    df = con.execute("SHOW tables").df()

print(df)

                     name
0                  events
1    fct_ab_buckets_daily
2  int_ab_events_bucketed
3           stg_event_log


In [4]:
%%sql

SELECT * FROM event_log


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 10:32:46,2,6927017134761466251,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 10:33:32,2,6927017134761466251,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 21:27:17,4,1241045378047175561,US,"{""num01"":""b""}",page_view,NaN
3,2025-01-01 21:30:00,4,1241045378047175561,US,"{""num01"":""b""}",watch,NaN
4,2025-01-01 21:30:55,4,1241045378047175561,US,"{""num01"":""b""}",page_view,NaN
...,...,...,...,...,...,...,...
1995146,2025-02-28 20:40:52,24996,968404688415237291,GB,"{""num01"":""a""}",watch,NaN
1995147,2025-02-28 16:19:09,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
1995148,2025-02-28 16:20:17,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
1995149,2025-02-28 16:20:30,25000,4860591953457782041,GB,"{""num01"":""a""}",watch,NaN


In [5]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    fct_ab_buckets_daily = con.execute("SELECT * FROM fct_ab_buckets_daily").df()


In [6]:
%%sql buckets <<

SELECT * FROM fct_ab_buckets_daily


,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
0,2025-01-04,DE,102,num01,a,43,38,4,0,1,6,6,3,0,1,24.07
1,2025-01-04,DE,17,num01,a,34,27,6,0,1,3,3,3,0,1,48.62
2,2025-01-04,US,68,num01,a,71,56,13,1,1,14,14,10,1,1,43.21
3,2025-01-04,GB,145,num01,a,18,13,3,1,1,5,5,3,1,1,19.57
4,2025-01-04,US,114,num01,a,54,44,9,1,0,10,10,8,1,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69483,2025-02-26,US,13,num01,b,1,1,0,0,0,1,1,0,0,0,0.00
69484,2025-02-26,DE,142,num01,b,5,5,0,0,0,1,1,0,0,0,0.00
69485,2025-02-26,GB,106,num01,b,7,6,1,0,0,1,1,1,0,0,0.00
69486,2025-02-26,GB,148,num01,b,6,6,0,0,0,1,1,0,0,0,0.00


In [7]:
buckets[
    (buckets.date_day=="2025-01-10")
    & (buckets.country=='US')
]

,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
1054,2025-01-10,US,196,num01,a,46,40,6,0,0,10,10,6,0,0,0.00
1056,2025-01-10,US,120,num01,a,59,50,8,1,0,12,12,7,1,0,0.00
1057,2025-01-10,US,180,num01,b,60,49,7,4,0,7,7,6,4,0,0.00
1060,2025-01-10,US,0,num01,b,69,56,9,2,2,11,11,7,2,2,82.68
1062,2025-01-10,US,3,num01,a,50,43,6,1,0,11,11,6,1,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61955,2025-01-10,US,12,num01,a,33,25,6,2,0,6,6,5,2,0,0.00
61960,2025-01-10,US,83,num01,b,33,25,5,1,2,6,6,5,1,2,163.88
66917,2025-01-10,US,32,num01,b,37,27,6,2,2,6,6,5,2,2,66.98
66941,2025-01-10,US,184,num01,b,31,26,4,0,1,7,7,4,0,1,327.82


In [8]:
(
    buckets.groupby('date_day')
    .total_unique_users.sum()
    .to_frame()
    .tail(20)
)
    

,total_unique_users
date_day,
2025-02-10,6305
2025-02-11,6062
2025-02-12,6119
2025-02-13,6673
2025-02-14,6401
2025-02-15,7321
2025-02-16,6962
2025-02-17,6228
2025-02-18,6667


# запуск экспериментов

In [9]:
from database import load_buckets
from analytics import run, significant_results, display_tt

buckets = load_buckets("num01")
results = run(buckets)
display_tt(significant_results(results))

,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect
0,total_events,a,b,996 926,998 225,194 069,191 270,1.544,3.58,1.591,0.247,2.840
1,purchase_amount,a,b,723 366,825 444,194 069,191 270,15.441,7.53,7.168,8.909,21.974
2,u_page_view,a,b,194 023,191 202,194 069,191 270,-0.011,1.99,0.021,-0.028,0.006
3,watch_count,a,b,128 953,136 607,194 069,191 270,7.393,17.22,1.701,6.060,8.727
4,u_watch,a,b,115 973,121 965,194 069,191 270,6.598,20.20,1.335,5.585,7.610
5,add_to_cart_count,a,b,22 310,23 544,194 069,191 270,7.040,6.36,3.983,3.628,10.451
6,u_add_to_cart,a,b,21 808,22 973,194 069,191 270,6.855,6.32,3.930,3.517,10.192
7,purchase_count,a,b,13 281,15 083,194 069,191 270,14.967,9.59,5.573,9.994,19.939
8,u_purchase,a,b,13 008,14 753,194 069,191 270,14.830,9.82,5.366,10.022,19.638


In [10]:
buckets["country"].value_counts().head(10)

country
US    23607
GB    23222
DE    22659
Name: count, dtype: int64

In [11]:
buckets_de = load_buckets("num01", country="DE")
results_de = run(buckets_de)
display_tt(significant_results(results_de))

,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect
0,purchase_amount,a,b,124 253,140 263,38 199,38 368,11.003,2.54,16.152,-2.633,24.638
1,u_page_view,a,b,38 197,38 352,38 199,38 368,-0.042,3.37,0.017,-0.079,-0.005
2,watch_count,a,b,24 449,26 501,38 199,38 368,7.537,6.64,4.549,4.005,11.068
3,u_watch,a,b,22 119,23 790,38 199,38 368,6.708,7.54,3.730,3.943,9.473
4,purchase_count,a,b,2 482,2 792,38 199,38 368,11.579,3.45,12.832,0.959,22.199
5,u_purchase,a,b,2 431,2 729,38 199,38 368,11.163,3.40,12.550,0.792,21.535
